<a href="https://colab.research.google.com/github/Git34Smakin/codez/blob/main/Detection_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import cv2
import numpy as np

class SimpleObjectDetector:
    def __init__(self):
        self.net = None
        self.classes = []
        self.load_model()

    def load_model(self):
        """Load pre-trained model using OpenCV DNN"""
        # Download these files:
        # - weights: https://pjreddie.com/media/files/yolov3.weights
        # - config: https://github.com/pjreddie/darknet/blob/master/cfg/yolov3.cfg?raw=true
        # - classes: https://github.com/pjreddie/darknet/blob/master/data/coco.names?raw=true

        try:
            self.net = cv2.dnn.readNet('yolov3.weights', 'yolov3.cfg')
            with open('coco.names', 'r') as f:
                self.classes = [line.strip() for line in f.readlines()]
        except:
            print("Model files not found. Please download YOLO weights and config.")

    def detect_objects_opencv(self, image_path, conf_threshold=0.5, nms_threshold=0.4):
        """Detect objects using OpenCV DNN"""
        image = cv2.imread(image_path)
        height, width = image.shape[:2]

        # Create blob from image
        blob = cv2.dnn.blobFromImage(image, 1/255.0, (416, 416), swapRB=True, crop=False)
        self.net.setInput(blob)

        # Get output layer names
        layer_names = self.net.getLayerNames()
        output_layers = [layer_names[i[0] - 1] for i in self.net.getUnconnectedOutLayers()]

        # Forward pass
        outputs = self.net.forward(output_layers)

        # Process detections
        boxes = []
        confidences = []
        class_ids = []

        for output in outputs:
            for detection in output:
                scores = detection[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]

                if confidence > conf_threshold:
                    center_x = int(detection[0] * width)
                    center_y = int(detection[1] * height)
                    w = int(detection[2] * width)
                    h = int(detection[3] * height)

                    x = int(center_x - w/2)
                    y = int(center_y - h/2)

                    boxes.append([x, y, w, h])
                    confidences.append(float(confidence))
                    class_ids.append(class_id)

        # Apply Non-Maximum Suppression
        indices = cv2.dnn.NMSBoxes(boxes, confidences, conf_threshold, nms_threshold)

        results = []
        if len(indices) > 0:
            for i in indices.flatten():
                x, y, w, h = boxes[i]
                results.append({
                    'class': self.classes[class_ids[i]],
                    'confidence': confidences[i],
                    'box': (x, y, x+w, y+h)
                })

                # Draw bounding box
                cv2.rectangle(image, (x, y), (x+w, y+h), (0, 255, 0), 2)
                label = f"{self.classes[class_ids[i]]}: {confidences[i]:.2f}"
                cv2.putText(image, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        return image, results

# Usage example for OpenCV version
def test_simple_detector():
    detector = SimpleObjectDetector()
    image_path = "test_image.jpg"

    result_image, detections = detector.detect_objects_opencv(image_path)

    cv2.imshow('Detection Results', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    print("Detections:")
    for detection in detections:
        print(f"- {detection['class']}: {detection['confidence']:.3f}")